# ThreePCF routes, spin, and cached results

This notebook starts from one projected angular bispectrum and demonstrates route planning, intermediate results, warm-up, and spin projection.

In [ ]:
import numpy as np
import fastnc
from fastnc.bispectrum import SPTMatterBispectrum3D
from fastnc.multipole import NumericMultipoleConfig
from fastnc.projection import LOSProjector
from fastnc.threepcf import ThreePCF, ThreePCFConfig

fastnc.configure_logging("INFO")
b3d = SPTMatterBispectrum3D.simple_debug()
b2d = LOSProjector.delta_like(z=0.5, chi=1300.0).project(b3d)
theta = np.geomspace(2.0e-3, 2.0e-2, 6)
phi = np.linspace(0.0, np.pi, 9)
config = ThreePCFConfig(
    spin=(0, 0, 0), Lmax=4, kmax=2,
    ell_min=20.0, ell_max=2000.0, n_ell=24,
    multipole=NumericMultipoleConfig(n_angle=65),
    use_coupling_cache=False,
)

## Inspect route assignment

The numeric route uses every term's numeric representation. Hybrid selects Slepian, then semi-analytic, then numeric fallback term by term. Planning performs no numerical evaluation.

In [ ]:
numeric = ThreePCF(config, b2d, theta, phi, route="numeric")
hybrid = ThreePCF(config, b2d, theta, phi, route="hybrid")
print(numeric.calculation_plan())
print(hybrid.calculation_plan())

With the optional graph dependencies and Graphviz dot installed, the same planner data can be rendered in a notebook.

In [ ]:
# hybrid.calculation_plan().graph(expand_terms=False)

## Inspect intermediate numerical levels

In [ ]:
multipoles = numeric.multipoles()
sample = multipoles.evaluate(
    mode=np.array([-1, 0, 1]),
    ell2=np.array([100.0, 200.0]),
    ell3=np.array([120.0, 240.0]),
)
hkernel = numeric.hkernel()
zetak = numeric.zetak()
zeta = numeric.zeta()
print(sample.shape, hkernel.values.shape, zetak.values.shape, zeta.values.shape)

HKernelTable remains on the dense internal ell grid. ZetaKTable is on the requested theta bins and retains opening-angle modes. ZetaTable adds the phi axis after resummation.

## Warm reusable structure before inference

Cold structured routes can spend appreciable time constructing universal radial kernels. warm_up() prepares reusable structure and clears its temporary physical prediction, so the next call recomputes model-dependent values while retaining warm structural resources.

In [ ]:
hybrid.warm_up()
# The next call represents a warm model evaluation.
zeta_hybrid = hybrid.zeta()

## Spin components and shear projection

In [ ]:
spin_config = ThreePCFConfig(
    spin=(2, 2, 2), Lmax=4, kmax=2,
    ell_min=20.0, ell_max=2000.0, n_ell=24,
    multipole=NumericMultipoleConfig(n_angle=65),
    use_coupling_cache=False,
)
spin_threepcf = ThreePCF(spin_config, b2d, theta, phi, route="hybrid")
epsilons = ((1, 1, -1), (1, -1, 1), (-1, 1, 1))
centroid = spin_threepcf.zeta(epsilons=epsilons, projection="centroid")
print(centroid.projection, centroid.values.shape)

The expensive cache stores the X projection. Centroid and orthocenter conventions are inexpensive phase conversions applied when returning the result. Epsilon choices that are not representatives for the configured spins raise an error rather than silently returning another component.

## Update target bins

set_phi preserves ZetaK and changes final resummation only. set_theta rebuilds the tuned FFTLog grid and radial results. Route can be changed explicitly with set_route; a new immutable numerical configuration requires a new ThreePCF instance.

In [ ]:
numeric.set_phi(np.linspace(0.0, np.pi, 13))
updated_phi = numeric.zeta()
numeric.set_theta(np.geomspace(1.0e-3, 3.0e-2, 8))
updated_theta = numeric.zeta()
print(updated_phi.values.shape, updated_theta.values.shape)